## Project Overview & Market Views

Current view of the market suggests the following return estimates:
  * Amazon (AMZN) 65%;
  * Bank of America (BAC) 0%;
  * Costco (COST) 35%;
  * Walt Disney (DIS) 5%;
  * Domino's Pizza (DPZ) 20,
  * Coke (KO) will go down by -5%;
  * McDonald's (MCD) -15%;
  * Microsoft (MSFT) 30%;
  * Nordic America Tankers (NAT) 80%;
  * Starbucks (SBUX) 5%.

Also we shall assign the following 68%-confidence intervals to views:
  * AMZN [0.5,  0.8]
  * BAC	[-0.1, 0.1]
  * COST [0.25, 0.45]
  * DIS	[-0.25, 0.25]
  * DPZ	[0.15, 0.25]
  * KO	[-0.1, 0]
  * MCD	[-0.3, 0]
  * MSFT [0.08, 0.40]
  * NAT	[0.1, 0.9]
  * SBUX [0, 0.3]

Using the Black-Litterman model, I determine a possible allocation strategy for investing in the previous ten assets (e.g. by using the Sharpe portfolio).
Comments are in the end.

Inputs:
  * market proxy and assets historical series
  * market capitalization of the assets
  * risk free rate for the considered period



In [7]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [9]:
import pandas as pd
import numpy as np
import requests
import json

#  1. Parameters
risk_free_rate = 0.02
frequency = 252
tau = 0.05

# 2. Data Sources

# The user provided this link for the data repository:
# https://github.com/vitas-g/black-litterman-optimization/tree/main/data
# To access raw data files, we convert the GitHub tree URL to a raw content URL.
github_repo_data_url = "https://github.com/vitas-g/black-litterman-optimization/tree/main/data"
base_url = github_repo_data_url.replace("github.com", "raw.githubusercontent.com").replace("/tree", "") + "/"

market_prices = pd.read_csv(base_url + "market_data.csv", index_col=0, parse_dates=True)
asset_prices = pd.read_csv(base_url + "assets_data.csv", index_col=0, parse_dates=True)
market_caps = pd.read_json("https://raw.githubusercontent.com/vitas-g/black-litterman-optimization/refs/heads/main/data/market_caps.json", orient='index').T

In [12]:
# Preparing Returns
asset_returns = asset_prices.pct_change().dropna()
market_returns = market_prices.pct_change().dropna()
print ()

I rewrote the part to read the market caps.

In [10]:
# Loading Market Caps (Robust Way)
# mkt_caps_data = response.json()

# clean_caps = {}
# for ticker, value in mkt_caps_data.items():
#     if isinstance(value, dict):
#         for inner_val in value.values():
#             if isinstance(inner_val, (int, float)):
#                 clean_caps[ticker] = inner_val
#                 break
#     else:
#         clean_caps[ticker] = value

market_caps = pd.read_json("https://raw.githubusercontent.com/vitas-g/black-litterman-optimization/refs/heads/main/data/market_caps.json", orient='index').T

#market_caps = pd.Series(clean_caps)
# CRITICAL: Align weights with returns columns
#market_caps = market_caps.reindex(asset_returns.columns)

market_weights = (market_caps['mkt_cap'] / market_caps['mkt_cap'].sum()).values

print (market_weights)
print("Data loaded and aligned successfully.")

[0.2778616739376415 0.04631757270120716 0.04813433917878655
 0.03142167649812369 0.0026657626571704542 0.05225296237931313
 0.04088248752060472 0.4781974421973414 0.00017368166104853735
 0.02209240126876287]
Data loaded and aligned successfully.


In [13]:
#  Market Equilibrium (Delta and Pi)
ann_market_return = market_returns.mean() * frequency
ann_market_var = market_returns.var() * frequency

# Risk aversion (Delta)
delta = (ann_market_return.item() - risk_free_rate) / ann_market_var.item()

#  Covariance Matrix
covariance_matrix = asset_returns.cov() * frequency

# Implied Equilibrium Returns (Pi)
pi = delta * covariance_matrix.dot(market_weights)

In [14]:
#  investor Views (P, Q, Omega)
q = np.array([0.65, 0.0, 0.35, 0.05, 0.20, -0.05, -0.15, 0.30, 0.80, 0.05])
p = np.eye(len(asset_returns.columns))

conf_intervals = [
    [0.5, 0.8], [ -0.1, 0.1], [0.25, 0.45], [-0.25, 0.25], [0.15, 0.25],
    [-0.1, 0.0], [-0.3, 0.0], [0.08, 0.40], [0.1, 0.9], [0.0, 0.3]
]

variances = []
for interval in conf_intervals:
    width = interval[1] - interval[0]
    sigma = width / 2
    variances.append(sigma**2)
omega = np.diag(variances)

#  Black-Litterman Calculations
tau_sigma = tau * covariance_matrix
A = p.dot(tau_sigma).dot(p.T) + omega

mu_bl = pi + tau_sigma.dot(p.T).dot(np.linalg.inv(A)).dot(q - p.dot(pi))

# Posterior Covariance (sigma_bl)
M = np.linalg.inv(np.linalg.inv(tau_sigma) + p.T.dot(np.linalg.inv(omega)).dot(p))
sigma_bl = covariance_matrix + M

In [15]:
# Portfolio Optimization (Sharpe)
excess_returns = mu_bl - risk_free_rate
weights_raw = np.linalg.inv(delta * sigma_bl).dot(excess_returns)
bl_weights = weights_raw / np.sum(weights_raw)

bl_weights_series = pd.Series(bl_weights, index=asset_returns.columns)

In [16]:
print (bl_weights)

[0.3993845833881585 -0.013970106914453042 0.1563610564232077
 0.009515448213357114 0.05894525326070523 -0.23160248061102667
 -0.0567277764716903 0.6461088411898339 0.01928710944341362
 0.01269807207849379]


In [17]:

#mu_bl = np.nan_to_num(mu_bl)
#sigma_bl = np.nan_to_num(sigma_bl)



print("Black-Litterman Optimized Weights ")
print(bl_weights_series.sort_values(ascending=False))

Black-Litterman Optimized Weights 
MSFT    0.646109
AMZN    0.399385
COST    0.156361
DPZ     0.058945
NAT     0.019287
SBUX    0.012698
DIS     0.009515
BAC     -0.01397
MCD    -0.056728
KO     -0.231602
dtype: object
